In [64]:
from Bio.PDB import PDBParser, PDBIO
import pandas as pd
import numpy as np
import os
import glob
import pickle
from umap import UMAP
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

# 1. Select residue from nsp9-10 (delete the rest)

In [2]:
def sele_resi(pdb_id, pdb_path, resi_chain, resi, resn, output_dir):
    '''pdb_id is the 4 lettel code (without.pdb)
    pdb_path is the fully path to the pdb file
    resi_chain = chain that the peptide is named as in the pdb
    resi = residue number of the peptide to be kept/selected
    resn = residue name (3 lettlers) of the peptide to be kept/selected
    output_dir = output_path for the pdb file to be outputed to (no need to include the file name) '''
    
    parser = PDBParser()
    structure = parser.get_structure(pdb_id, pdb_path)

    resi_dele = []

    for residue in structure.get_residues():
        chain = residue.get_parent()

        if chain.id == resi_chain and residue.id[1] == int(resi):
            if residue.id[2] == resn:
                continue
            else:
                    print("ERROR! residue inputed does not match residue in pdb")
        elif chain.id == resi_chain:
            resi_dele.append(residue.id)
        else:
             continue
    
    #deleted none selected residues within seleted peptide
    peptide_chain = structure[0][resi_chain]
    for residue in resi_dele:
        peptide_chain.detach_child(residue)
    
    #save output
    output_name = f"{pdb_id}_{resi_chain}_{resn}{resi}.pdb"
    output_path = os.path.join(output_dir, output_name)

    io = PDBIO()
    io.set_structure(structure)
    io.save(output_path)

loop through and output into 7TA4_single_resi file

In [4]:
pdb_path = "/Users/hannah/Documents/Study/DPhil/Y1/Rotation 1/CoV-Mpro/scripts/Mpro-nsp/7TA4.pdb"
pdb_id = "7TA4"
output_dir = "/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi"

#getting resi and resn information
df = pd.read_csv("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_nsp9-10_resi_info.csv")

for _, row in df.iterrows():
    resi = row["resi"]
    resn = row["resn"]

    sele_resi(pdb_id, pdb_path, "D", resi, resn, output_dir)
  

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5304.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 5668.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 6038.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 6044.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5304.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv

ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb


/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5304.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 5668.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 6038.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 6044.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5304.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv

ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb
ERROR! residue inputed does not match residue in pdb


/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5304.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 5668.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 6038.
  warnings.warn(
/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 6044.
  warnings.warn(


## 1.2 getting apo 7TA4 in solution
apo at chain B (i.e. deleted chain D peptide) - water intacted so I can still call chain "D" in process_pdb

In [91]:
parser = PDBParser()
structure = parser.get_structure("7TA4", "/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4.pdb")

chain_D_resi = []

for resi in structure.get_residues():
    chain = resi.get_parent()

    if chain.id == "D":
        if resi.id[0] == "W":
            continue
        else: 
            chain_D_resi.append(resi.id)

peptide_chain = structure[0]["D"]
for residue in chain_D_resi:
    peptide_chain.detach_child(residue)

io = PDBIO()
io.set_structure(structure)
io.save("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo.pdb")

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning:


/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning:


/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning:


/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning:




# 2. Parse ATOMICA required data_index_file

In [10]:
pdb_dir = "/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi"

df = []

for pdb in glob.glob(os.path.join(pdb_dir, '*.pdb')):
    pdb_path = pdb
    pdb_id = os.path.basename(pdb)
    
    df.append({
        "pdb_id": pdb_id,
        "pdb_path": pdb_path,
        "chain1": "B",
        "chain2": "D",
        "lig_code": "",
        "lig_smiles": "",
        "lig_resi": ""
    })

output_df = pd.DataFrame(df)
output_path = os.path.join(pdb_dir, "7TA4_single_resi_atomica_index.csv")
output_df.to_csv(output_path, index=False)


## 2.2 Getting csv for soluble apo 7TA4

did manually
path = "/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo_atomica_index.csv" 

# 3. run process_pdb.py from ATOMICA

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/data/process_pdbs.py 
--data_index_file /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_atomica_index.csv
--out_path /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/data/process_pdbs.py --data_index_file /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_atomica_index.csv --out_path /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl

save as 7TA4_single_resi_processed.pkl

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/data/process_pdbs.py 
--data_index_file /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo_atomica_index.csv
--out_path /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo_processed.pkl

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/data/process_pdbs.py --data_index_file /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo_atomica_index.csv --out_path /Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_D_apo_processed.pkl

# 4.Get embeddings from ATOMICA

model 1

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py
--model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v1.json
--model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v1.pt
--data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl
--output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding1.pkl

as a paste-able line

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py --model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v1.json --model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v1.pt --data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl --output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding1.pkl

model 2

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py
--model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v2.json
--model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v2.pt
--data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl
--output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding2.pkl

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py --model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v2.json --model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v2.pt --data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl --output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding2.pkl


model 3

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py
--model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v3.json
--model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v3.pt
--data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl
--output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding3.pkl

In [ ]:
python /Users/hannah/code/rotation1/ATOMICA/get_embeddings.py --model_config ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v3.json --model_weights ~/code/rotation1/ATOMICA/checkpoints/ATOMICA_checkpoints/prot_interface/atomica_interface_v3.pt --data_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_processed.pkl --output_path ~/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding3.pkl

read everything and store a mean

In [27]:
with open("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding1.pkl", "rb") as f:
    em1 = pickle.load(f)

with open("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding2.pkl", "rb") as f:
    em2 = pickle.load(f)

with open("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_single_resi_embedding3.pkl", "rb") as f:
    em3 = pickle.load(f)

graph_em1_single_resi = np.array([e['graph_embedding'] for e in em1 if e['id'] != "7TA4.pdb_B_D"]) #### single out the full complex for the rest of these
graph_em2_single_resi = np.array([e['graph_embedding'] for e in em2 if e['id'] != "7TA4.pdb_B_D"])
graph_em3_single_resi = np.array([e['graph_embedding'] for e in em3 if e['id'] != "7TA4.pdb_B_D"])

graph_em_single_resi_mean = np.mean(np.stack([graph_em1_single_resi, graph_em2_single_resi, graph_em3_single_resi]), axis = 0)


In [30]:
graph_em1_full = [e['graph_embedding'] for e in em1 if e['id'] == "7TA4.pdb_B_D"]
graph_em2_full = [e['graph_embedding'] for e in em2 if e['id'] == "7TA4.pdb_B_D"]
graph_em3_full = [e['graph_embedding'] for e in em3 if e['id'] == "7TA4.pdb_B_D"]

graph_em_full_mean = np.mean(np.stack([graph_em1_full, graph_em2_full, graph_em3_full]), axis = 0)

In [31]:
print(graph_em_full_mean)

[[ 0.03749669  0.13570067 -0.00818682 -0.0314607  -0.04974402 -0.08662837
   0.05677481 -0.14938395 -0.24809353 -0.11199903  0.30589417  0.06015372
  -0.04754477  0.0249453  -0.08118087 -0.19542186  0.00500363 -0.10917163
  -0.09882015  0.26841486 -0.10186046 -0.01394444 -0.15654935  0.05688372
   0.11259302 -0.13128689  0.2269091  -0.10947306 -0.10759882 -0.20775299
   0.1411959  -0.11726654]]


In [ ]:
#checking that all of the ids are in the same order actually

ids_1 = [e['id'] for e in em1]
ids_2 = [e['id'] for e in em2]
ids_3 = [e['id'] for e in em3]

print(ids_1 == ids_2)
print(ids_2 == ids_3)

#the mean should be fine then

True
True


In [41]:
id_single_resi = [e['id'] for e in em1 if e['id'] != "7TA4.pdb_B_D"]
id_full = "7TA4.pdb_B_D"


# 5. Generate UMAP

In [ ]:
reducer = UMAP(random_state=42, n_neighbors=30)
umap_embedding = reducer.fit_transform(graph_em_single_resi_mean)

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/hannah/miniforge3/envs/atomicaenv/lib/python3.11/site-packages/umap/umap_.py:2462: UserWarning:

n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1



In [6]:
label_info = pd.read_csv("/Users/hannah/code/rotation1/rotation1_code/7TA4_single_resi/7TA4_nsp9-10_resi_info.csv")

In [42]:
plot = pd.DataFrame({
    "x": umap_embedding[:, 0],
    "y": umap_embedding[:, 1],
    "pdb_id": id_single_resi
})

In [51]:
fig = px.scatter(plot, x = "x", y="y", hover_data=["pdb_id"], title="nsp9-10 single residue UMAP", height=500, width=500)

fig.update_traces(marker=dict(size=10, opacity=0.7))
fig.show()

In [48]:
umap_full_em = reducer.transform(graph_em_full_mean)

In [54]:
plot_full_em = pd.DataFrame({
    "x": umap_full_em[:, 0],
    "y": umap_full_em[:, 1],
    "pdb_id": id_full
})

In [59]:
fig.add_trace(go.Scatter(x = umap_full_em[:, 0], y = umap_full_em[:, 1]))

position of full complex differ, maybe UMAP is not the best representation to compare these

# 6. Cosine similarity

In [74]:
sum_single_resi_em = np.sum(np.stack(graph_em_single_resi_mean), axis = 0) #sum along the rows

In [63]:
print(sum_single_resi_em)

[ 2.4174871   0.26292714 -0.40624726  0.34634152 -0.39341843 -1.4851599
  2.4907944   0.3049766  -0.78595084 -0.07746623  0.31538224  2.3476079
 -1.5532968  -0.7912923   1.2702415  -1.1084174   1.894469   -1.2453256
  0.4030813  -0.5357729  -1.5651617   0.26873398 -2.1325645   1.6702999
  0.00572221  0.61324847  0.07235748 -0.34578198 -1.6512817  -2.2240965
  0.00531285  0.76394266]


In [75]:
sum_single_resi_em = np.reshape(sum_single_resi_em,(1, -1))
graph_em_full_mean = np.reshape(graph_em1_full, (1, -1))

In [76]:
full_vs_single_resi = cosine_similarity(graph_em_full_mean, sum_single_resi_em)

In [77]:
print(full_vs_single_resi)

[[0.14442399]]
